# LoRA SigLIP-2 + Orientation Head + OCFR (Kaggle GPU)

Trains the large SigLIP-2 (LoRA) + orientation head + OCFR + attribute classifier end-to-end.

**Setup:**
1. Kaggle **Dataset** with 3 files: `PA100K.zip`, `train_lora.py`, `train_lora_ocfr.py`.
2. Right panel -> **Add Data** -> attach it.
3. Right panel -> **Settings**: Accelerator = **GPU T4 x2**, Internet = **On**.
4. Run cells 1-5. If the smoke test prints an mA -> use **Save Version -> Save & Run All** to run the full job overnight in the background.

In [ ]:
# 1. deps
!pip install -q peft
!pip uninstall -y torchao

In [ ]:
# 2. locate dataset, copy scripts, unzip data
import os, shutil, zipfile
INP = '/kaggle/input/' + os.listdir('/kaggle/input')[0]
print('using:', INP, '->', os.listdir(INP))
for f in ['train_lora.py', 'train_lora_ocfr.py']:
    shutil.copy(f'{INP}/{f}', f'/kaggle/working/{f}')
with zipfile.ZipFile(f'{INP}/PA100K.zip') as z:
    z.extractall('/kaggle/working/data')
!find /kaggle/working/data -maxdepth 3 -type d

In [ ]:
# 3. paths
ANN = '/kaggle/working/data/PA-100K/annotation/annotation.mat'
IMG = '/kaggle/working/data/PA-100K/data/release_data/release_data'
import os; assert os.path.exists(ANN) and os.path.isdir(IMG), 'check paths vs tree above'
print('OK')

In [ ]:
# 4. SMOKE TEST (2000 imgs, 1 epoch) -- confirms the full OCFR pipeline runs
%cd /kaggle/working
!python train_lora_ocfr.py --ann "{ANN}" --img_dir "{IMG}" --epochs 1 --batch 32 --limit_train 2000 --out /kaggle/working

In [ ]:
# 5. FULL run (all 80k, 6 epochs). Use 'Save Version -> Save & Run All' to run overnight.
%cd /kaggle/working
!python train_lora_ocfr.py --ann "{ANN}" --img_dir "{IMG}" --epochs 6 --batch 32 --out /kaggle/working